# Aufbau einer wiederverwendbaren aktuariellen Tarifierungsbibliothek mit PROC FCMP

## Zusammenfassung

Ein Schaden- und Unfallversicherer kapselt seine zentrale Tarifierungsmathematik — reine Prämie, Kosten-/Risikozuschlag, Glaubwürdigkeitsmischung nach der Methode der begrenzten Fluktuation und diskontierte Rückstellungsschätzung — als benutzerdefinierte Funktionen und eine Subroutine mit mehreren Rückgabewerten in **PROC FCMP**. Die kompilierte Bibliothek wird über die Systemoption `CMPLIB=` registriert und anschließend zeilenweise aus einem DATA-Step aufgerufen, der ein synthetisches Portfolio von 100 Policen bewertet. Jede bewertete Kennzahl in diesem Notebook — der Glaubwürdigkeitsfaktor `z`, die gemischte reine Prämie, die berechnete Prämie und der barwertige Schadenrückstellungsbetrag — wird von diesen kompilierten Routinen erzeugt, nicht durch inline-Arithmetik. Das Ergebnis: Die implizite Schadenquote liegt bei 60,5 % (ländlich), 55,8 % (vorstädtisch) und 63,6 % (städtisch) — in jedem Segment komfortabel unter 100 %, was bestätigt, dass die zugeschlagene Prämie den erwarteten Schaden deckt, während der Tarifierungsschritt sauber und nachvollziehbar bleibt.

## Datenquellen

| Datensatz | Zeilen | Beschreibung | Schlüsselvariablen |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Synthetisches, bestehendes Schaden-/Unfallversicherungsportfolio, inline mit `rand()` erzeugt | `policy_id`, `region` (städtisch/vorstädtisch/ländlich), `years_insured`, `exposure` (Fahrzeugjahre), `claim_count` (Poisson), `incurred_loss` (Gamma-Schadenhöhe x Anzahl), `class_pure_prem` (manueller Tarif nach Region) |

Der DATA-Step durchläuft einen größeren `policy_id`-Bereich, aber diese Umgebung läuft unlizenziert, sodass die Ausgabe auf die ersten **100 Beobachtungen** begrenzt ist — das bewertete Portfolio unten spiegelt diese 100 Policen wider.

# Aufbau einer wiederverwendbaren aktuariellen Tarifierungsbibliothek mit PROC FCMP

Aktuare wiederholen dieselben Berechnungen über Tarifierungs-, Reservierungs- und Berichtsaufgaben hinweg: Schäden in eine *reine Prämie* umrechnen, *Kosten- und Risikozuschläge* anwenden, um zu einer berechneten Prämie zu gelangen, die eigene Erfahrung einer einzelnen Police mit einem Klassentarif mittels *Glaubwürdigkeit* mischen und künftige Zahlungsströme auf den Barwert *diskontieren*. Das erneute Eintippen dieser Formeln in jedem DATA-Step lädt zu Copy-Paste-Fehlern ein und macht Änderungen an Annahmen mühsam.

**PROC FCMP** (der SAS-Funktionscompiler) erlaubt es uns, jede Formel einmal als benannte Funktion oder Subroutine zu definieren, die kompilierten Routinen in einer Bibliothek zu speichern und sie dann wie jede eingebaute SAS-Funktion aufzurufen. In diesem Notebook werden wir:

1. Eine kleine aktuarielle Funktionsbibliothek mit `PROC FCMP` kompilieren.
2. Sie für die Sitzung mit der Systemoption `CMPLIB=` registrieren.
3. Ein synthetisches Schaden- und Unfallversicherungsportfolio erzeugen.
4. Jede Police bewerten, indem wir unsere benutzerdefinierten Funktionen und eine Subroutine mit mehreren Rückgabewerten aus einem einzigen DATA-Step aufrufen.
5. Das bewertete Portfolio zusammenfassen und interpretieren.

## Schritt 1 — Ein synthetisches Policenportfolio erzeugen

Wir simulieren einen Bestand laufender Kfz-Policen (die ersten 100 werden unten unter der Unlizenziert-Modus-Grenze bewertet). Jede Police gehört zu einer Tarifregion mit ihrer eigenen manuellen *reinen Prämie* (erwarteter Schaden pro Fahrzeugjahr). Schadenanzahlen folgen einem durch die Exposure skalierten Poisson-Prozess, und Schadenhöhen folgen einer Gammaverteilung, sodass `incurred_loss` ein realistischer zusammengesetzter (Poisson x Gamma) Schaden ist. `years_insured` bestimmt später das Glaubwürdigkeitsgewicht. Ein fester Seed über `call streaminit` macht die Daten reproduzierbar.

In [1]:
DATEN portfolio;
    AUFRUFEN streaminit(20260531);
    LÄNGE region $14;
    AUSFÜHRUNG policy_id = 10001 BIS 12000;
        /* Tarifregion zuweisen */
        u = rand('uniform');
        WENN u < 0.40 DANN AUSFÜHRUNG; region = 'städtisch';    base_pp = 820; lambda = 0.18; ENDE;
        SONST WENN u < 0.75 DANN AUSFÜHRUNG; region = 'vorstädtisch'; base_pp = 540; lambda = 0.11; ENDE;
        SONST AUSFÜHRUNG; region = 'ländlich';    base_pp = 360; lambda = 0.07; ENDE;

        /* Versicherungsdauer (Jahre) und Exposure (Fahrzeugjahre) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Zusammengesetzter Schadenprozess: Poisson-Frequenz, Gamma-Schadenhöhe */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        AUSFÜHRUNG c = 1 BIS claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        ENDE;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manuelle Klassen-Reinprämie für die Exposure dieser Police */
        class_pure_prem = round(base_pp * exposure, 0.01);

        AUSGABE;
    ENDE;
    BEHALTEN policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
    BEZEICHNUNG
        policy_id       = 'Policennummer'
        region          = 'Region'
        years_insured   = 'Versicherungsjahre'
        exposure        = 'Exposure (Fahrzeugjahre)'
        claim_count     = 'Schadenanzahl'
        incurred_loss   = 'Angefallener Schaden'
        class_pure_prem = 'Tarifreine Prämie (Klasse)';
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=portfolio(obs=8) noobs;
    TITEL 'Erste 8 simulierte Policen';
AUSFÜHREN;

                                               Erste 8 simulierte Policen                                               

policy_id         region  years_insured  exposure  claim_count  incurred_loss  class_pure_prem
    10001  ländlich                   5      1.36            0              0            489.6
    10002  städtisch                  8      0.97            1        3432.28            795.4
    10003  städtisch                  2      1.53            2        7155.92           1254.6
    10004  vorstädtisch               9       2.4            0              0             1296
    10005  ländlich                   5      0.99            0              0            356.4
    10006  vorstädtisch               1      0.83            0              0            448.2
    10007  ländlich                   5      0.97            0              0            349.2
    10008  ländlich                   7      2.32            0              0            835.2

... 92 more observatio


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.45 seconds
  cpu   0.45 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Schritt 2 — Die aktuarielle Funktionsbibliothek kompilieren

Nun das Herzstück des Notebooks. `PROC FCMP` mit `OUTLIB=work.actfuncs.pricing` kompiliert vier Funktionen und eine Subroutine in das `pricing`-Paket des Datensatzes `work.actfuncs`:

- **`pure_premium`** — beobachteter Schaden pro Einheit Exposure (Frequenz x Schadenhöhe kombiniert).
- **`credibility_z`** — Glaubwürdigkeitsfaktor nach der Methode der begrenzten Fluktuation `Z = sqrt(n / (n + k))`, wobei `n` die Exposure-Jahre der Police und `k` eine Anpassungskonstante ist.
- **`charged_premium`** — wendet einen proportionalen Risikozuschlag und eine feste Kostenquote auf einen Schadenkostensatz an, um zur tatsächlich berechneten Prämie zu gelangen.
- **`pv_reserve`** — Barwert einer künftigen Schadenzahlung, `FV / (1+r)**t`, verwendet zur Diskontierung von Schadenrückstellungen.
- **`blend_premium`** (eine SUBROUTINE) — verwendet `OUTARGS`, um *zwei* Werte gleichzeitig zurückzugeben: die glaubwürdigkeitsgewichtete reine Prämie und den dabei verwendeten Glaubwürdigkeitsfaktor, sodass der DATA-Step beide in einem einzigen Aufruf erhält.

Jede Routine wird mit `ENDSUB` abgeschlossen, und die Subroutine benennt ihre beschreibbaren Argumente mit `OUTARGS`.

In [2]:
PROZEDUR fcmp outlib=work.actfuncs.pricing;

    /* Beobachtete reine Prämie: Schadenkostensatz pro Einheit Exposure */
    function pure_premium(loss, exposure);
        WENN exposure <= 0 DANN RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Glaubwürdigkeit nach begrenzter Fluktuation: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        WENN n_years <= 0 DANN RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Berechnete Prämie = Schadenkostensatz hochgerechnet für Risikozuschlag + Kosten */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        WENN expense_ratio >= 1 DANN RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Barwert einer künftigen Schadenzahlung, diskontiert über t Jahre zum Zinssatz r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Glaubwürdigkeitsmischung: gibt die gewichtete reine Prämie UND das verwendete Z zurück */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

AUSFÜHREN;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Schritt 3 — Die Bibliothek mit CMPLIB= registrieren

Das Kompilieren der Routinen allein genügt nicht; SAS muss mitgeteilt werden, wo sie zu finden sind, wenn ein späterer DATA-Step oder eine Prozedur einen Namen referenziert, den sie nicht als eingebaut erkennt. Die Systemoption `CMPLIB=` verweist auf den Datensatz (nicht den dreistufigen Paketnamen), der den kompilierten Code enthält. Nach dieser `OPTIONS`-Anweisung ist jede der obigen Funktionen und Subroutinen namentlich aufrufbar.

In [3]:
OPTIONEN cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Schritt 4 — Das Portfolio mit den benutzerdefinierten Funktionen bewerten

Der Tarifierungs-DATA-Step ist nun fast frei von Arithmetik — die aktuarielle Absicht liest sich direkt aus den Funktionsnamen ab. Für jede Police:

1. Berechnen wir die eigene beobachtete reine Prämie der Police mit `pure_premium`.
2. Rufen wir die Subroutine `blend_premium` auf, um sie glaubwürdigkeitsgewichtet mit dem regionalen Klassentarif zu mischen, wobei sowohl der gemischte Schadenkostensatz als auch der Glaubwürdigkeitsfaktor `z` über `OUTARGS` zurückgegeben werden.
3. Rechnen wir den gemischten Schadenkostensatz mit einem Risikozuschlag von 12 % und einer Kostenquote von 25 % über `charged_premium` zur berechneten Prämie hoch.
4. Schätzen wir die noch offene Schadenrückstellung als 35 % des angefallenen Schadens der Police und diskontieren sie drei Jahre bei 4 % auf den Barwert mit `pv_reserve`.

Beachten Sie, dass die Ausgabeargumente der Subroutine (`blended_pp`, `z`) vor dem `CALL` initialisiert werden müssen. Der Rückstellungsbarwert variiert von Police zu Police, da er vom tatsächlichen angefallenen Schaden jeder Police abhängt — schadenfreie Policen tragen eine Rückstellung von null, sodass auch ihr `reserve_pv` null ist.

In [4]:
DATEN rated;
    FESTLEGEN portfolio;

    /* 1. Eigene Schadenerfahrung der Police als reine Prämie */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Eigene Erfahrung glaubwürdigkeitsgewichtet mit dem Klassentarif mischen.
          k = 4 Exposure-Jahre für nahezu volle Glaubwürdigkeit. */
    blended_pp = .;
    z = .;
    AUFRUFEN blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Gemischten Schadenkostensatz (pro Fahrzeugjahr) in berechnete Prämie umrechnen */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Noch offene Schadenrückstellung = 35 % des angefallenen Schadens, erwartete
          Abwicklung in 3 Jahren; auf den Barwert bei 4 % diskontieren. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    BEHALTEN policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
    BEZEICHNUNG
        own_pp       = 'Eigene reine Prämie'
        blended_pp   = 'Gemischte reine Prämie'
        z            = 'Glaubwürdigkeitsfaktor Z'
        premium      = 'Prämie'
        case_reserve = 'Schadenrückstellung'
        reserve_pv   = 'Barwert Rückstellung';
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=rated(obs=10) noobs;
    TITEL 'Bewertetes Portfolio (erste 10 Policen)';
    VAR policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
AUSFÜHREN;

                                        Bewertetes Portfolio (erste 10 Policen)                                         

policy_id         region  years_insured  exposure   own_pp  blended_pp      z  premium  reserve_pv
    10001  ländlich                   5      1.36        0       91.67  0.745   186.18           0
    10002  städtisch                  8      0.97  3538.43     3039.59  0.816  4402.95     1067.95
    10003  städtisch                  2      1.53  4677.07     3046.88  0.577  6961.51     2226.55
    10004  vorstädtisch               9       2.4        0       90.69  0.832   325.03           0
    10005  ländlich                   5      0.99        0       91.67  0.745   135.52           0
    10006  vorstädtisch               1      0.83        0       298.5  0.447   369.98           0
    10007  ländlich                   5      0.97        0       91.67  0.745   132.79           0
    10008  ländlich                   7      2.32        0       72.82  0.798   252.29


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Schritt 5 — Das bewertete Buch zusammenfassen

Da jede Police über dieselbe kompilierte Bibliothek bepreist wurde, können wir das Portfolio nach Region aggregieren. Wir berichten die mittlere berechnete Prämie, den mittleren Glaubwürdigkeitsfaktor, den gesamten angefallenen Schaden und die gesamte barwertige Schadenrückstellung und leiten dann die implizite Schadenquote je Segment ab, um zu sehen, ob die zugeschlagene Prämie den erwarteten Schaden deckt.

In [5]:
PROZEDUR MITTELWERTE DATEN=rated mean sum maxdec=2 nonobs;
    KLASSE region;
    VAR premium z incurred_loss reserve_pv;
    TITEL 'Portfolio-Zusammenfassung nach Tarifregion';
AUSFÜHREN;

/* Implizite Schadenquote = angefallener Schaden / berechnete Prämie, plus die
   barwertige Rückstellung, die das Segment trägt, nach Region. */
PROZEDUR SQL;
    TITEL 'Implizite Schadenquote und Rückstellungsbarwert nach Region';
    AUSWÄHLEN region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     format=dollar12.2,
           sum(premium)                          AS total_premium  format=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     format=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve format=dollar12.2
    VON rated
    GROUP NACH region
    ORDER NACH region;
QUIT;
TITEL;

                                       Portfolio-Zusammenfassung nach Tarifregion                                       

                                                  The MEANS Procedure

                                          Analysis Variable : premium Prämie

        Region                  Mean            Sum
        -------------------------------------------
        ländlich              476.61       11438.62
        städtisch            1987.17       67563.93
        vorstädtisch          813.04       34147.74
        -------------------------------------------

                                    Analysis Variable : z Glaubwürdigkeitsfaktor Z

        Region                  Mean            Sum
        -------------------------------------------
        ländlich                0.71          17.14
        städtisch               0.70          23.90
        vorstädtisch            0.68          28.36
        -------------------------------------------

                   


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretation der Ergebnisse

Der Tarifierungs-DATA-Step schreibt nirgends eine einzelne Diskontierungs- oder Glaubwürdigkeitsformel inline aus — er ruft lediglich `pure_premium`, `blend_premium`, `charged_premium` und `pv_reserve` auf. Das ist der Gewinn von **PROC FCMP**: Die aktuariellen Annahmen leben in einer einzigen kompilierten Bibliothek, die unit-getestet, versionskontrolliert und über Tarifierungs-, Reservierungs- und Berichtsaufgaben hinweg wiederverwendet werden kann. Das Ändern der Glaubwürdigkeitskonstante `k`, des Risikozuschlags oder der Kostenquote ist eine einzige Änderung in der Bibliothek, keine Suche durch jedes Programm.

Beim Lesen des bewerteten Buchs und der regionalen Aggregation:

- **Glaubwürdigkeit (`z`)** steigt mit `years_insured`, genau wie es die Formel der begrenzten Fluktuation `Z = sqrt(n / (n + k))` vorschreibt. Unter den ersten zehn Policen erzielt die einjährige vorstädtische Police (10006) `z = 0,447`, die zweijährige städtische Police (10003) `z = 0,577`, während die neunjährige vorstädtische Police (10004) `z = 0,832` erreicht. Policen mit dünner Erfahrung werden zum regionalen Klassentarif hingezogen; langjährige stützen sich auf ihre eigenen Schäden.
- **Mischung in Aktion:** Schadenfreie Policen (der Großteil des Buchs) haben `own_pp = 0`, sodass `blend_premium` eine `blended_pp` nahe `(1 - z)` mal dem Klassentarif zurückgibt — z. B. landet Police 10001 (ländlich, `z = 0,745`) bei `91,67` gegenüber einem Klassentarif von `360`/Fahrzeugjahr. Die zwei städtischen Policen, die tatsächlich Schäden hatten, 10002 und 10003, ziehen ihren gemischten Schadenkostensatz stattdessen zu ihrer eigenen hohen Erfahrung hoch (`3039,59` und `3046,88`).
- **Die berechnete Prämie** liegt über dem gemischten Schadenkostensatz, weil `charged_premium` einen Risikozuschlag von 12 % hinzufügt und für eine Kostenquote von 25 % hochrechnet, ein fester Multiplikator von `1,12 / 0,75 = 1,493` auf den Schadenkostensatz. Die mittlere berechnete Prämie liegt bei `476,61` (ländlich), `813,04` (vorstädtisch) und `1.987,17` (städtisch).
- **Diskontierte Rückstellungen:** `pv_reserve` diskontiert die noch offene Schadenrückstellung jeder Police (35 % des angefallenen Schadens) über drei Jahre bei 4 %, ein Barwertfaktor von `0,889`. Schadenfreie Policen tragen `reserve_pv = 0`; die beiden städtischen Schadenfälle steuern `1067,95` und `2226,55` bei. Aggregiert hält das Buch eine barwertige Rückstellung von `2.151,56 $` (ländlich), `5.932,52 $` (vorstädtisch) und `13.364,13 $` (städtisch).
- **Implizite Schadenquoten** liegen bei 60,5 % (ländlich), 55,8 % (vorstädtisch) und 63,6 % (städtisch) — alle komfortabel unter 100 %, sodass die zugeschlagene Prämie in jedem Segment den erwarteten Schaden deckt. Das städtische Segment läuft am heißesten, getrieben von seinen beiden großen simulierten Schäden; eine echte Tarifüberprüfung würde testen, ob sich dieses Signal über weitere Schadenjahre hinweg bestätigt, bevor der manuelle Tarif angepasst wird.

Die Subroutine `blend_premium` demonstriert außerdem das `OUTARGS`-Idiom zur Rückgabe mehrerer Ergebnisse aus einem einzigen `CALL` — der gemischte Schadenkostensatz und der Glaubwürdigkeitsfaktor `z` kommen gemeinsam zurück — wodurch separate Funktionsaufrufe vermieden werden und die policenweise Tarifierungslogik kompakt und nachvollziehbar bleibt.